# Lab 06 External V2 — 02 Fact Encounters

**Dataset:** Synthea Healthcare  
**Architecture:** External Delta tables  
**Compute:** Databricks Serverless compatible

## Purpose

Build `fact_encounters` at **one row per healthcare encounter**, resolve
foreign keys to the external Gold dimensions, persist the result at an explicit
external Delta location, and reconcile the output to the Synthea source.

> Serverless compatibility rule: this notebook does **not** call
> `REFRESH TABLE`, `CACHE TABLE`, `UNCACHE TABLE`, or Spark cache-refresh APIs.


## 1. Runtime context

In [ ]:
def ensure_text_widget(name: str, default: str, label: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)


def ensure_dropdown_widget(
    name: str,
    default: str,
    choices: list[str],
    label: str,
) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default, choices, label)


ensure_text_widget("catalog", "dbr_dev", "01 Catalog")
ensure_text_widget("source_schema", "parvinbadalov", "02 Source schema")
ensure_text_widget(
    "source_volume_name",
    "lab06_gold_analytics",
    "03 Source volume",
)
ensure_text_widget(
    "target_schema",
    "parvinbadalov_lab06_ext",
    "04 Target schema",
)
ensure_text_widget(
    "external_gold_root",
    "REPLACE_WITH_EXTERNAL_GOLD_ROOT",
    "05 External Gold root",
)
ensure_dropdown_widget(
    "run_validation",
    "true",
    ["true", "false"],
    "06 Run validation",
)

catalog = dbutils.widgets.get("catalog").strip()
source_schema = dbutils.widgets.get("source_schema").strip()
source_volume_name = dbutils.widgets.get("source_volume_name").strip()
target_schema = dbutils.widgets.get("target_schema").strip()
external_gold_root = dbutils.widgets.get("external_gold_root").strip().rstrip("/")
run_validation = (
    dbutils.widgets.get("run_validation").strip().lower() == "true"
)

if not external_gold_root or external_gold_root == "REPLACE_WITH_EXTERNAL_GOLD_ROOT":
    raise ValueError(
        "external_gold_root must be passed by lab06_00_dev_runner "
        "or entered manually."
    )

if not external_gold_root.lower().startswith("abfss://"):
    raise ValueError("external_gold_root must be an abfss:// path.")

source_volume_path = (
    f"/Volumes/{catalog}/{source_schema}/{source_volume_name}"
)
source_csv_path = f"{source_volume_path}/source/csv"
reference_path = f"{source_volume_path}/reference"
target_schema_fqn = f"{catalog}.{target_schema}"

print(f"Catalog            : {catalog}")
print(f"Source schema      : {source_schema}")
print(f"Source volume      : {source_volume_name}")
print(f"Target schema      : {target_schema}")
print(f"External Gold root : {external_gold_root}")
print(f"Run validation     : {run_validation}")

## 2. Serverless-safe helpers and table names

In [ ]:
import sys
from pathlib import Path

from pyspark.sql import functions as F

current_dir = Path.cwd()
lab_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))

from src.external_tables import (
    normalize_location,
    overwrite_external_delta,
    registered_table_location,
    register_external_delta_table,
    validate_registered_location,
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema_fqn}")

print("Serverless-safe external-table helpers loaded.")

TABLES = {
    "dim_date": f"{target_schema_fqn}.dim_date",
    "dim_patient": f"{target_schema_fqn}.dim_patient",
    "dim_provider": f"{target_schema_fqn}.dim_provider",
    "dim_organization": f"{target_schema_fqn}.dim_organization",
    "dim_payer": f"{target_schema_fqn}.dim_payer",
    "fact_encounters": f"{target_schema_fqn}.fact_encounters",
}

fact_location = f"{external_gold_root}/fact_encounters"
encounters_source = f"{source_csv_path}/encounters.csv"

for name in [
    "dim_date",
    "dim_patient",
    "dim_provider",
    "dim_organization",
    "dim_payer",
]:
    if not spark.catalog.tableExists(TABLES[name]):
        raise RuntimeError(f"Required dimension is missing: {TABLES[name]}")

print(f"Source : {encounters_source}")
print(f"Target : {TABLES['fact_encounters']}")
print(f"Path   : {fact_location}")

## 3. Load and validate encounter source

In [ ]:
encounters_src = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(encounters_source)
)

REQUIRED_COLUMNS = {
    "Id", "START", "STOP", "PATIENT", "ORGANIZATION", "PROVIDER", "PAYER",
    "ENCOUNTERCLASS", "CODE", "DESCRIPTION", "BASE_ENCOUNTER_COST",
    "TOTAL_CLAIM_COST", "PAYER_COVERAGE", "REASONCODE", "REASONDESCRIPTION",
}

missing_columns = sorted(REQUIRED_COLUMNS - set(encounters_src.columns))
if missing_columns:
    raise ValueError(
        "encounters.csv is missing required columns: "
        + ", ".join(missing_columns)
    )

print(f"Source rows: {encounters_src.count():,}")
display(encounters_src.limit(10))

## 4. Prepare measures and business columns

In [ ]:
prepared_encounters = (
    encounters_src
    .select(
        F.col("Id").alias("encounter_id"),
        F.to_timestamp("START").alias("encounter_start_ts"),
        F.to_timestamp("STOP").alias("encounter_stop_ts"),
        F.col("PATIENT").alias("patient_id"),
        F.col("ORGANIZATION").alias("organization_id"),
        F.col("PROVIDER").alias("provider_id"),
        F.col("PAYER").alias("payer_id"),
        F.col("ENCOUNTERCLASS").alias("encounter_class"),
        F.col("CODE").alias("encounter_code"),
        F.col("DESCRIPTION").alias("encounter_description"),
        F.col("BASE_ENCOUNTER_COST").cast("decimal(18,2)").alias("base_encounter_cost"),
        F.col("TOTAL_CLAIM_COST").cast("decimal(18,2)").alias("total_claim_cost"),
        F.col("PAYER_COVERAGE").cast("decimal(18,2)").alias("payer_coverage"),
        F.col("REASONCODE").alias("reason_code"),
        F.col("REASONDESCRIPTION").alias("reason_description"),
    )
    .withColumn("encounter_date", F.to_date("encounter_start_ts"))
    .withColumn(
        "duration_minutes",
        (
            F.unix_timestamp("encounter_stop_ts")
            - F.unix_timestamp("encounter_start_ts")
        ) / F.lit(60.0),
    )
    .withColumn(
        "patient_responsibility",
        (
            F.col("total_claim_cost") - F.col("payer_coverage")
        ).cast("decimal(18,2)"),
    )
)

display(prepared_encounters.limit(10))

## 5. Source-grain validation

In [ ]:
source_quality = (
    prepared_encounters
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("encounter_id").alias("distinct_ids"),
        F.sum(F.when(F.col("encounter_id").isNull(), 1).otherwise(0)).alias("null_ids"),
        F.sum(F.when(F.col("encounter_start_ts").isNull(), 1).otherwise(0)).alias("bad_start"),
        F.sum(F.when(F.col("encounter_stop_ts").isNull(), 1).otherwise(0)).alias("bad_stop"),
        F.sum(F.when(F.col("duration_minutes") < 0, 1).otherwise(0)).alias("negative_duration"),
        F.sum(F.when(F.col("total_claim_cost") < 0, 1).otherwise(0)).alias("negative_claim"),
    )
    .first()
)

source_failures = []
if source_quality["row_count"] != source_quality["distinct_ids"]:
    source_failures.append("duplicate encounter_id")
for field in ["null_ids", "bad_start", "bad_stop", "negative_duration", "negative_claim"]:
    if (source_quality[field] or 0) > 0:
        source_failures.append(field)

display(spark.createDataFrame(
    [(
        int(source_quality["row_count"]),
        int(source_quality["distinct_ids"]),
        int(source_quality["null_ids"] or 0),
        int(source_quality["bad_start"] or 0),
        int(source_quality["bad_stop"] or 0),
        int(source_quality["negative_duration"] or 0),
        int(source_quality["negative_claim"] or 0),
        "PASS" if not source_failures else "FAIL",
    )],
    ["rows","distinct_ids","null_ids","bad_start","bad_stop",
     "negative_duration","negative_claim","status"],
))

if run_validation and source_failures:
    raise RuntimeError("Encounter source validation failed: " + ", ".join(source_failures))

## 6. Resolve dimension keys

In [ ]:
dim_date = spark.table(TABLES["dim_date"]).select("date_key", "full_date")
dim_patient = spark.table(TABLES["dim_patient"]).select("patient_key", "patient_id")
dim_provider = spark.table(TABLES["dim_provider"]).select("provider_key", "provider_id")
dim_organization = spark.table(TABLES["dim_organization"]).select(
    "organization_key", "organization_id"
)
dim_payer = spark.table(TABLES["dim_payer"]).select("payer_key", "payer_id")

fact_encounters_df = (
    prepared_encounters.alias("e")
    .join(dim_date.alias("d"), F.col("e.encounter_date") == F.col("d.full_date"), "left")
    .join(dim_patient.alias("pat"), F.col("e.patient_id") == F.col("pat.patient_id"), "left")
    .join(dim_provider.alias("pr"), F.col("e.provider_id") == F.col("pr.provider_id"), "left")
    .join(
        dim_organization.alias("org"),
        F.col("e.organization_id") == F.col("org.organization_id"),
        "left",
    )
    .join(dim_payer.alias("pay"), F.col("e.payer_id") == F.col("pay.payer_id"), "left")
    .select(
        F.xxhash64("e.encounter_id").alias("encounter_key"),
        F.col("e.encounter_id"),
        F.col("d.date_key"),
        F.col("pat.patient_key"),
        F.col("pr.provider_key"),
        F.col("org.organization_key"),
        F.col("pay.payer_key"),
        F.col("e.patient_id"),
        F.col("e.provider_id"),
        F.col("e.organization_id"),
        F.col("e.payer_id"),
        F.col("e.encounter_start_ts"),
        F.col("e.encounter_stop_ts"),
        F.col("e.encounter_date"),
        F.col("e.encounter_class"),
        F.col("e.encounter_code"),
        F.col("e.encounter_description"),
        F.col("e.base_encounter_cost"),
        F.col("e.total_claim_cost"),
        F.col("e.payer_coverage"),
        F.col("e.patient_responsibility"),
        F.round(F.col("e.duration_minutes"), 2).alias("duration_minutes"),
        F.col("e.reason_code"),
        F.col("e.reason_description"),
    )
)

display(fact_encounters_df.limit(10))

## 7. Foreign-key validation

In [ ]:
fk = (
    fact_encounters_df
    .agg(
        F.sum(F.when(F.col("date_key").isNull(), 1).otherwise(0)).alias("date"),
        F.sum(F.when(F.col("patient_key").isNull(), 1).otherwise(0)).alias("patient"),
        F.sum(F.when(F.col("organization_key").isNull(), 1).otherwise(0)).alias("organization"),
        F.sum(F.when(F.col("payer_key").isNull(), 1).otherwise(0)).alias("payer"),
        F.sum(
            F.when(
                F.col("provider_id").isNotNull() & F.col("provider_key").isNull(),
                1,
            ).otherwise(0)
        ).alias("provider"),
    )
    .first()
)

fk_failures = [
    name for name in ["date", "patient", "organization", "payer", "provider"]
    if (fk[name] or 0) > 0
]

display(spark.createDataFrame(
    [(name, int(fk[name] or 0), "PASS" if (fk[name] or 0) == 0 else "FAIL")
     for name in ["date","patient","organization","payer","provider"]],
    ["foreign_key","missing_rows","status"],
))

if run_validation and fk_failures:
    raise RuntimeError("Fact FK validation failed: " + ", ".join(fk_failures))

## 8. Persist external `fact_encounters`

In [ ]:
overwrite_external_delta(
    spark,
    fact_encounters_df,
    TABLES["fact_encounters"],
    fact_location,
)

print(f"Created: {TABLES['fact_encounters']}")
print(f"Location: {registered_table_location(spark, TABLES['fact_encounters'])}")

## 9. Reconciliation and completion

In [ ]:
source_count = prepared_encounters.count()

target = spark.table(TABLES["fact_encounters"])
profile = target.agg(
    F.count("*").alias("rows"),
    F.countDistinct("encounter_id").alias("ids"),
    F.countDistinct("encounter_key").alias("keys"),
).first()

grain_ok = (
    int(profile["rows"]) == source_count
    and int(profile["rows"]) == int(profile["ids"])
    and int(profile["rows"]) == int(profile["keys"])
)

source_fin = prepared_encounters.agg(
    *[F.sum(c).alias(c) for c in [
        "base_encounter_cost", "total_claim_cost",
        "payer_coverage", "patient_responsibility"
    ]]
).first()
target_fin = target.agg(
    *[F.sum(c).alias(c) for c in [
        "base_encounter_cost", "total_claim_cost",
        "payer_coverage", "patient_responsibility"
    ]]
).first()

financial_failures = [
    c for c in [
        "base_encounter_cost", "total_claim_cost",
        "payer_coverage", "patient_responsibility"
    ]
    if source_fin[c] != target_fin[c]
]

final_checks = [
    ("source_grain", len(source_failures) == 0),
    ("foreign_keys", len(fk_failures) == 0),
    ("fact_grain", grain_ok),
    ("financial_reconciliation", len(financial_failures) == 0),
    ("external_location",
     normalize_location(registered_table_location(spark, TABLES["fact_encounters"]))
     == normalize_location(fact_location)),
]

display(spark.createDataFrame(
    [(n, "PASS" if ok else "FAIL") for n, ok in final_checks],
    ["validation_area","status"],
))

failed = [n for n, ok in final_checks if not ok]
if run_validation and failed:
    raise RuntimeError("02 Fact Encounters failed: " + ", ".join(failed))

print("LAB 06 EXTERNAL V2 — FACT ENCOUNTERS COMPLETE")
print("Serverless compatibility: PASS")
print("REFRESH TABLE calls: 0")
print("Next: lab06_03_fact_conditions")